In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

sys.path.append("src")
from entropy_pruning import (
    AttentionForecaster,
    MLPForecaster,
    ConvForecaster,
    UNILoRAClassifier,
    build_attention_cache,
    build_loaders,
    set_seed,
    train_forecaster,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

/home/vcivale/miniconda3/envs/entropy_pruning/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


In [ ]:
CFG = dict(
    data_dir="/raid/DATASETS/NCT-CRC-HE",
    img_size=224,
    batch_size=64,
    num_workers=0,
    seed=42,
    layer_source=2,
    layer_target=23,
    epochs=30,
    lr=1e-4,
    weight_decay=0.05,
)

# --- Architecture configurations ---
ARCH_CONFIGS = [
    # (name, model_class, kwargs)
    ("Attention h=128 L=1", AttentionForecaster, dict(hidden=128, n_heads=4, n_layers=1, dropout=0.1)),
    ("Attention h=256 L=1", AttentionForecaster, dict(hidden=256, n_heads=4, n_layers=1, dropout=0.1)),
    ("Attention h=128 L=2", AttentionForecaster, dict(hidden=128, n_heads=4, n_layers=2, dropout=0.1)),
    ("Attention h=256 L=2", AttentionForecaster, dict(hidden=256, n_heads=4, n_layers=2, dropout=0.1)),
    ("Attention h=256 L=3", AttentionForecaster, dict(hidden=256, n_heads=4, n_layers=3, dropout=0.1)),
    ("MLP h=256 L=2",       MLPForecaster,       dict(hidden=256, n_layers=2, dropout=0.1)),
    ("MLP h=256 L=3",       MLPForecaster,       dict(hidden=256, n_layers=3, dropout=0.1)),
    ("MLP h=512 L=3",       MLPForecaster,       dict(hidden=512, n_layers=3, dropout=0.1)),
    ("Conv h=256 L=2 k=5",  ConvForecaster,      dict(hidden=256, n_layers=2, dropout=0.1, kernel_size=5)),
    ("Conv h=256 L=3 k=5",  ConvForecaster,      dict(hidden=256, n_layers=3, dropout=0.1, kernel_size=5)),
    ("Conv h=256 L=2 k=9",  ConvForecaster,      dict(hidden=256, n_layers=2, dropout=0.1, kernel_size=9)),
]

set_seed(CFG["seed"])
dataset_name = Path(CFG["data_dir"]).name
classifier_ckpt = Path(f"/raid/DATASETS/checkpoints-Attention-Pruning/{dataset_name}/uni_finetuned/best_model.pt")
cache_path = Path(f"/raid/DATASETS/NCT-CRC-HE/data_cache/{dataset_name}_forecaster_dataset.h5")
forecaster_dir = Path(f"/raid/DATASETS/checkpoints-Attention-Pruning//{dataset_name}/arch_ablation")
forecaster_dir.mkdir(parents=True, exist_ok=True)

print(f"Architectures: {len(ARCH_CONFIGS)}")
for name, cls, kw in ARCH_CONFIGS:
    print(f"  {name} ({cls.__name__})")

Architectures: 11
  Attention h=128 L=1 (AttentionForecaster)
  Attention h=256 L=1 (AttentionForecaster)
  Attention h=128 L=2 (AttentionForecaster)
  Attention h=256 L=2 (AttentionForecaster)
  Attention h=256 L=3 (AttentionForecaster)
  MLP h=256 L=2 (MLPForecaster)
  MLP h=256 L=3 (MLPForecaster)
  MLP h=512 L=3 (MLPForecaster)
  Conv h=256 L=2 k=5 (ConvForecaster)
  Conv h=256 L=3 k=5 (ConvForecaster)
  Conv h=256 L=2 k=9 (ConvForecaster)


In [3]:
loaders = build_loaders(
    data_dir=CFG["data_dir"],
    img_size=CFG["img_size"],
    batch_size=CFG["batch_size"],
    num_workers=CFG["num_workers"],
    drop_last_train=False,
)

model = UNILoRAClassifier(loaders.n_classes).to(device)
model.load_state_dict(torch.load(classifier_ckpt, map_location=device), strict=False)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

if not cache_path.exists():
    build_attention_cache(
        model=model,
        loaders={"train": loaders.train_loader, "val": loaders.val_loader, "test": loaders.test_loader},
        device=device,
        source_layers=[CFG["layer_source"]],
        target_layers=[CFG["layer_target"]],
        save_path=cache_path,
    )
print("Cache:", cache_path)

del model
torch.cuda.empty_cache()

cache:test: 100%|██████████| 113/113 [01:31<00:00,  1.24it/s]


Cache: /raid/DATASETS/NCT-CRC-HE/data_cache/NCT-CRC-HE_forecaster_dataset.h5


In [ ]:
all_results = []
for i, (arch_name, model_cls, model_kwargs) in enumerate(ARCH_CONFIGS):
    tag = arch_name.replace(" ", "_").replace("=", "")
    print(f"\n[{i+1}/{len(ARCH_CONFIGS)}] {arch_name}")

    save_path = forecaster_dir / f"forecaster_{tag}.pt"

    forecaster_model = model_cls(embed_dim=1024, **model_kwargs)
    n_params = sum(p.numel() for p in forecaster_model.parameters())
    print(f"  params: {n_params:,}")

    result = train_forecaster(
        h5_cache_path=cache_path,
        layer_source=CFG["layer_source"],
        layer_target=CFG["layer_target"],
        device=device,
        epochs=CFG["epochs"],
        lr=CFG["lr"],
        weight_decay=CFG["weight_decay"],
        save_path=save_path,
        model=forecaster_model,
    )

    result["arch_name"] = arch_name
    result["arch_class"] = model_cls.__name__
    result["n_params"] = n_params
    all_results.append(result)

    print(f"  rho={result['test_rho_forecaster']:.4f}  "
          f"val_kl={result['best_val_kl']:.4f}")

    del result["model"]
    torch.cuda.empty_cache()


[1/11] Attention h=128 L=1
  params: 363,393


  0%|          | 0/1407 [00:00<?, ?it/s]

 83%|████████▎ | 1173/1407 [1:21:12<1:59:47, 30.72s/it]

In [ ]:
df = pd.DataFrame([
    {
        "architecture": r["arch_name"],
        "class": r["arch_class"],
        "n_params": r["n_params"],
        "best_val_kl": r["best_val_kl"],
        "best_val_rho": r["best_val_rho"],
        "test_rho_forecaster": r["test_rho_forecaster"],
        "test_rho_token_norm": r["test_rho_token_norm"],
        "delta_rho": r["test_rho_forecaster"] - r["test_rho_token_norm"],
    }
    for r in all_results
])
df = df.sort_values("test_rho_forecaster", ascending=False).reset_index(drop=True)
Path("results").mkdir(exist_ok=True)
df.to_csv(f"results/arch_ablation_{dataset_name}.csv", index=False)
df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Bar chart: rho per architecture
bars = axes[0].barh(df["architecture"], df["test_rho_forecaster"], color="steelblue")
axes[0].set_xlabel("Spearman \u03c1 (test)")
axes[0].set_title("Architecture comparison")
axes[0].bar_label(bars, fmt="%.4f", padding=3)
axes[0].invert_yaxis()
axes[0].grid(alpha=0.3, axis="x")

# 2. Bar chart: delta_rho (improvement over norm baseline)
bars2 = axes[1].barh(df["architecture"], df["delta_rho"], color="darkorange")
axes[1].set_xlabel("\u0394\u03c1 (forecaster \u2212 norm baseline)")
axes[1].set_title("Improvement over token-norm baseline")
axes[1].bar_label(bars2, fmt="%.4f", padding=3)
axes[1].invert_yaxis()
axes[1].grid(alpha=0.3, axis="x")

# 3. Scatter: params vs rho
for cls_name, color in [("AttentionForecaster", "steelblue"), ("MLPForecaster", "seagreen"), ("ConvForecaster", "coral")]:
    mask = df["class"] == cls_name
    axes[2].scatter(df.loc[mask, "n_params"], df.loc[mask, "test_rho_forecaster"],
                    label=cls_name, color=color, s=80, edgecolors="k", linewidths=0.5)
axes[2].set_xlabel("# Parameters")
axes[2].set_ylabel("Spearman \u03c1 (test)")
axes[2].set_title("Params vs Performance")
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"results/arch_ablation_{dataset_name}.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
print("Risultati ablation architettura:\n")
print(df[["architecture", "class", "n_params", "test_rho_forecaster",
          "test_rho_token_norm", "delta_rho", "best_val_kl"]].to_string(index=False))